In [ ]:
from transformers import pipeline, WhisperForConditionalGeneration, WhisperProcessor
import evaluate
from datasets import load_dataset

import torch

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
data = load_dataset('brunopbb/ufcg-labmet-fala-texto-validation', split='test+train')
metric = evaluate.load('wer')

In [8]:
data

Dataset({
    features: ['audio', 'transcription'],
    num_rows: 69
})

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor


def evaluate_model(model_name, validation_data):
    
    processor = WhisperProcessor.from_pretrained(model_name)
    model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)

    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id
    model.config.pad_token_id = model.config.eos_token_id   

    predictions = []
    references = []

    for audio_file, reference_text in zip(validation_data['audio'], validation_data['transcription']):
        
        inputs = processor(
            audio_file['array'], 
            return_tensors="pt", 
            sampling_rate=16000
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}  # Mover tensores para GPU

        
        generated_ids = model.generate(inputs["input_features"])
        transcriptions = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

        
        predictions.append(transcriptions)
        references.append(reference_text)

    
    
    metric = evaluate.load("wer")
    wer = metric.compute(predictions=predictions, references=references)
    return wer


In [13]:
evaluate_model('/mnt/hd/v2-medium/whisper_finetuned-medium-v2', data)

0.20324189526184538

In [12]:
evaluate_model('openai/whisper-medium', data)

Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


0.30423940149625933